# 01 — Data Inventory

Creates an inventory of all files under `data/raw` and summarizes file counts,
extensions and sizes.

In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh


In [2]:
import pandas as pd

records = []
for path in sorted(RAW_DIR.rglob("*")):
    if path.is_file():
        records.append({
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "folder": str(path.parent.relative_to(RAW_DIR)),
            "name": path.name,
            "extension": path.suffix.lower(),
            "size_mb": round(path.stat().st_size / (1024 ** 2), 4),
        })

inventory = pd.DataFrame(records)
display(inventory.head(20))
print(f"Total files: {len(inventory)}")

,relative_path,folder,name,extension,size_mb
0,data\raw\boundary\Khulna.cpg,boundary,Khulna.cpg,.cpg,0.0000
1,data\raw\boundary\Khulna.dbf,boundary,Khulna.dbf,.dbf,0.0025
2,data\raw\boundary\Khulna.prj,boundary,Khulna.prj,.prj,0.0001
3,data\raw\boundary\Khulna.sbn,boundary,Khulna.sbn,.sbn,0.0001
4,data\raw\boundary\Khulna.sbx,boundary,Khulna.sbx,.sbx,0.0001
5,data\raw\boundary\Khulna.shp,boundary,Khulna.shp,.shp,0.1037
6,data\raw\boundary\Khulna.shx,boundary,Khulna.shx,.shx,0.0001
7,data\raw\gauge\bmd_monthly_rainfall_2017_2022.csv,gauge,bmd_monthly_rainfall_2017_2022.csv,.csv,0.0221
8,data\raw\precipitation\CCS\2017_01.tif,precipitation\CCS,2017_01.tif,.tif,0.0021
9,data\raw\precipitation\CCS\2017_02.tif,precipitation\CCS,2017_02.tif,.tif,0.0017


Total files: 849


In [3]:
summary = (
    inventory.groupby(["folder", "extension"], dropna=False)
    .agg(file_count=("name", "count"), total_size_mb=("size_mb", "sum"))
    .reset_index()
    .sort_values(["folder", "extension"])
)

output = INTERIM_DIR / "data_inventory.csv"
inventory.to_csv(output, index=False)
display(summary)
print(f"Saved: {output}")

,folder,extension,file_count,total_size_mb
0,boundary,.cpg,1,0.0000
1,boundary,.dbf,1,0.0025
2,boundary,.prj,1,0.0001
3,boundary,.sbn,1,0.0001
4,boundary,.sbx,1,0.0001
5,boundary,.shp,1,0.1037
6,boundary,.shx,1,0.0001
7,gauge,.csv,1,0.0221
8,precipitation\CCS,.tif,72,0.1228
9,precipitation\CDR,.tif,72,0.0648


Saved: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\interim\data_inventory.csv
